In [ ]:
import pandas as pd
import httpx

In [ ]:
df = pd.read_csv(r"C:\Users\Manideep S\OneDrive - COGNINE\POC-AI-Powered Mental Health Prediction using PHQ-9 (DSM-5)\tools\misc\phq9_consultations_with_unique_notes.csv")

In [ ]:
df.head(1)

In [ ]:
df["response"].head(1)

In [ ]:
type(df["response"] if isinstance(df["response"], dict) else df["parsed_response"].head(1))

In [ ]:
import requests
def register_user(
    user_id:str,
    emailid: str,
    username: str,
    firstname: str,
    lastname: str,
    age: str,
    gender: str,
    industry: str,
    profession: str,
    password: str,
    api_url: str = "http://localhost:5000/api/register"
):
    payload = {
        "user_id": user_id,
        "emailid": emailid,
        "username": username,
        "firstname": firstname,
        "lastname": lastname,
        "age": age,
        "gender": gender,
        "industry": industry,
        "profession": profession,
        "password": password,
        "confirm_password": password
    }
    response = requests.post(api_url, json=payload)
    print(response)

In [ ]:
def worker(row):
    return register_user(
        user_id=str(row["id"]),
        emailid=row["emailid"],
        username=row["username"],
        firstname=row["firstname"],
        lastname=row["lastname"],
        age=str(row["age"]),
        gender=row["gender"],
        industry=row["industry"],
        profession=row["profession"],
        password=row["password"]
    )
# worker(df.iloc[0])

In [ ]:
import asyncio
import httpx
import pandas as pd
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
import time

nest_asyncio.apply()  # Needed for Jupyter or nested event loops

# -------------------------
# Configuration
# -------------------------
API_URL = "http://localhost:5000/api/register"
MAX_CONCURRENT = 8       # Max simultaneous requests
MAX_RETRIES = 3          # Retry failed requests
RETRY_DELAY = 2          # Seconds between retries

# -------------------------
# Async API call function
# -------------------------
async def register_user_async(row, client):
    """
    Registers a single user.
    Supports optional 'id' field for CSV import.
    Retries failed requests up to MAX_RETRIES times.
    """
    payload = {
        "emailid": row["emailid"],
        "username": row["username"],
        "firstname": row["firstname"],
        "lastname": row["lastname"],
        "age": str(row["age"]),
        "gender": row["gender"],
        "industry": row["industry"],
        "profession": row["profession"],
        "password": row["password"],
        "confirm_password": row["password"]
    }
    if "id" in row:
        payload["user_id"] = str(row["id"])

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = await client.post(API_URL, json=payload, timeout=30)
            if response.status_code >= 400:
                error_msg = f"Error {response.status_code} for {row['emailid']}: {response.text}"
                if attempt == MAX_RETRIES:
                    print(error_msg)
                    return {"emailid": row["emailid"], "error": response.text, "status": response.status_code}
                else:
                    await asyncio.sleep(RETRY_DELAY)
                    continue
            return response.json()
        except httpx.RequestError as e:
            if attempt == MAX_RETRIES:
                print(f"Request error for {row['emailid']}: {e}")
                return {"emailid": row["emailid"], "error": str(e)}
            await asyncio.sleep(RETRY_DELAY)

# -------------------------
# Worker function with semaphore
# -------------------------
async def worker(row, client, semaphore):
    async with semaphore:
        return await register_user_async(row, client)

# -------------------------
# Main processing function
# -------------------------
async def process_dataframe(df, max_concurrent=MAX_CONCURRENT):
    semaphore = asyncio.Semaphore(max_concurrent)

    # Use single shared AsyncClient for all requests
    async with httpx.AsyncClient(timeout=30) as client:
        tasks = [worker(row, client, semaphore) for _, row in df.iterrows()]

        results = []
        # tqdm_asyncio.as_completed allows dynamic progress bar
        for f in tqdm_asyncio.as_completed(tasks, total=len(tasks), desc="Processing users"):
            result = await f
            results.append(result)
        return results

# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":

    start_time = time.time()
    results = asyncio.run(process_dataframe(df, max_concurrent=8))
    print(f"\nCompleted in {time.time() - start_time:.2f} seconds")
    print(results)


In [ ]:
# Async timeout handling with await and retries

import json
import ast
import aiohttp
import asyncio
import pandas as pd
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
import numpy as np

nest_asyncio.apply()


def safe_int(val, default=0):
    try:
        return int(val)
    except (ValueError, TypeError):
        return default


# Helper: ensure dict parsing
def ensure_dict(value, user_id=None):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            try:
                return ast.literal_eval(value)
            except Exception:
                with open("bad_responses.log", "a", encoding="utf-8") as f:
                    f.write(f"User {user_id} -> {value}\n")
                return {}
    return {}


# Helper: convert all values in dict to JSON serializable types
def make_serializable(d):
    if isinstance(d, dict):
        return {k: make_serializable(v) for k, v in d.items()}
    elif isinstance(d, (np.integer, int)):
        return int(d)
    elif isinstance(d, (np.floating, float)):
        return float(d)
    elif isinstance(d, (np.bool_, bool)):
        return bool(d)
    elif d is None:
        return None
    else:
        return d


# Worker: async API call with retries
async def worker(session, row, semaphore, max_retries=3):
    url = "http://localhost:5000/api/phq9"  # Flask endpoint
    payload = {
        "user_id": str(row.get("user_id") or ""),
        "responses": make_serializable(
            ensure_dict(row.get("response") or row.get("parsed_response"), row.get("user_id"))
        ),
        "totalScore": safe_int(row.get("total_score")),
        "doctors_notes": str(row.get("doctor_notes") or ""),
        "patients_notes": str(row.get("patient_notes") or "")
    }

    for attempt in range(1, max_retries + 1):
        try:
            async with semaphore:  # limit concurrent requests
                async with session.post(url, json=payload) as resp:
                    if resp.status != 200:
                        text = await resp.text()
                        print(f"❌ Error {resp.status} for user_id {row.get('user_id')}: {text}")
                        return None
                    return await resp.json()

        except Exception as e:
            print(f"⚠️ Client exception (attempt {attempt}) for user_id {row.get('user_id')}: {e}")

        await asyncio.sleep(2 * attempt)  # backoff before retry

    return None


# Main async loop
async def main(df, concurrency=10):
    timeout = aiohttp.ClientTimeout(total=60)  # 60 sec timeout
    connector = aiohttp.TCPConnector(limit=50)  # connection pool limit

    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        semaphore = asyncio.Semaphore(concurrency)  # max concurrent tasks
        tasks = [worker(session, df.iloc[i], semaphore) for i in range(len(df))]
        results = []

        for f in tqdm_asyncio.as_completed(tasks, total=len(tasks), desc="Processing users"):
            result = await f
            if result:
                results.append(result)

        return results


# Run async function in Jupyter
results = asyncio.get_event_loop().run_until_complete(main(df, concurrency=10))
print("✅ Completed")
print(results)


In [ ]:
import json
import ast
import aiohttp
import asyncio
import pandas as pd
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
import numpy as np

nest_asyncio.apply()

def safe_int(val, default=0):
    try:
        return int(val)
    except (ValueError, TypeError):
        return default


# Helper: ensure dict parsing
def ensure_dict(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            try:
                return ast.literal_eval(value)
            except Exception:
                with open("bad_responses.log", "a", encoding="utf-8") as f:
                    f.write(f"Bad response -> {value}\n")
                return {}
    return {}

# Helper: convert all values in dict to JSON serializable types
def make_serializable(d):
    if isinstance(d, dict):
        return {k: make_serializable(v) for k, v in d.items()}
    elif isinstance(d, (np.integer, int)):
        return int(d)
    elif isinstance(d, (np.floating, float)):
        return float(d)
    elif isinstance(d, (np.bool_, bool)):
        return bool(d)
    elif d is None:
        return None
    else:
        return d

# Worker: async API call using user_id
async def worker(session, row):
    url = "http://localhost:5000/api/phq9"  # Flask endpoint
    user_id = safe_int(row.get("user_id"))
    
    if user_id == 0:  # skip rows without valid user_id
        print(f"⚠️ Skipping row, missing user_id: {row.get('username')}")
        return None

    payload = {
        "user_id": user_id,  # use user_id directly
        "responses": make_serializable(
            ensure_dict(row.get("response") or row.get("parsed_response"))
        ),
        "totalScore": safe_int(row.get("total_score")),
        "doctors_notes": str(row.get("doctor_notes") or ""),
        "patients_notes": str(row.get("patient_notes") or "")
    }

    try:
        async with session.post(url, json=payload) as resp:
            if resp.status != 200:
                text = await resp.text()
                print(f"❌ Error {resp.status} for user_id {user_id}: {text}")
                return None
            return await resp.json()
    except Exception as e:
        print(f"⚠️ Exception for user_id {user_id}: {e}")
        return None

# Main async loop
async def main(df):
    async with aiohttp.ClientSession() as session:
        tasks = [worker(session, df.iloc[i]) for i in range(len(df))]
        results = []
        for f in tqdm_asyncio.as_completed(tasks, total=len(tasks), desc="Processing users"):
            result = await f
            if result:
                results.append(result)
        return results


# Run async function in Jupyter
results = asyncio.get_event_loop().run_until_complete(main(df))
print("✅ Completed")
print(results)
